In [1]:
# Using PyTorch 1.4

import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import random
import torch
import fire
import logging
import os
import csv
import re
import shutil

from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from torch.optim import SGD
from tqdm import tqdm, trange
import torch.nn.functional as F

E0000 00:00:1740454105.629534   11363 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1740454105.634208   11363 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
conda install git

Channels:
 - conda-forge
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: | 

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Install LM-Eval
%pip install git+https://github.com/EleutherAI/lm-evaluation-harness.git

  Cloning https://github.com/EleutherAI/lm-evaluation-harness.git to /tmp/pip-req-build-vcw34f41
  Running command git clone --filter=blob:none --quiet https://github.com/EleutherAI/lm-evaluation-harness.git /tmp/pip-req-build-vcw34f41
  Resolved https://github.com/EleutherAI/lm-evaluation-harness.git to commit 2b2fa97b284602a44796194419a6ca19701bda33
  Installing build dependencies ... one
  Getting requirements to build wheel ... one
  Preparing metadata (pyproject.toml) ... one
Note: you may need to restart the kernel to use updated packages.


In [4]:

YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
  - medmcqa
  - medqa_4options
  - task: mmlu_anatomy
    task_alias: "anatomy (mmlu)"
  - task: mmlu_clinical_knowledge
    task_alias: "clinical_knowledge (mmlu)"
  - task: mmlu_college_medicine
    task_alias: "college_medicine (mmlu)"
  - task: mmlu_medical_genetics
    task_alias: "medical_genetics (mmlu)"
  - task: mmlu_professional_medicine
    task_alias: "professional_medicine (mmlu)"
  - task: mmlu_college_biology
    task_alias: "college_biology (mmlu)"
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [5]:
class CSVMIMIC(Dataset):
    
    def __init__(self, control_code, truncate=False, gpt2_type="gpt2", max_length=768):

        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        self.notes = []

        # This uses the MIMIC3 csv
        
        with open('MIMIC_NoteEvents_deanonymized.csv', newline='', encoding='ISO-8859-1') as csvfile:
            tweet_csv = csv.reader(csvfile)
            for row in tweet_csv:
                self.notes.append(torch.tensor(
                    self.tokenizer.encode(f"<|{control_code}|>{row[10][:max_length]}<|endoftext|>")
                ))
                
        if truncate:
            self.notes = self.notes[:20000]
        self.notes_count = len(self.notes)
        
    def __len__(self):
        return self.notes_count

    def __getitem__(self, item):
        return self.notes[item]



In [6]:
def pack_tensor(new_tensor, packed_tensor, max_seq_len):
    if packed_tensor is None:
        return new_tensor, True, None
    if new_tensor.size()[1] + packed_tensor.size()[1] > max_seq_len:
        return packed_tensor, False, new_tensor
    else:
        packed_tensor = torch.cat([new_tensor, packed_tensor[:, 1:]], dim=1)
        return packed_tensor, True, None

In [7]:
def train(
    dataset,
    model,
    tokenizer,
    batch_size=16,
    epochs=4,
    lr=2e-5,
    max_seq_len=400,
    warmup_steps=5000,
    gpt2_type="gpt2",
    device="cuda",
    output_dir=".",
    output_prefix="wreckgar",
    test_mode=False,
    save_model_on_epoch=False,
):

    acc_steps = 100

    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=-1
    )

    train_dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    accumulating_batch_count = 0
    input_tensor = None

    for epoch in range(epochs):

        print(f"Training epoch {epoch}")
        for idx, entry in tqdm(enumerate(train_dataloader)):
            (input_tensor, carry_on, remainder) = pack_tensor(entry, input_tensor, 768)

            if carry_on and idx != len(train_dataloader) - 1:
                continue

            input_tensor = input_tensor.to(device)
            outputs = model(input_tensor, labels=input_tensor)
            loss = outputs[0]
            loss.backward()

            if (accumulating_batch_count % batch_size) == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                model.zero_grad()

            accumulating_batch_count += 1
            input_tensor = None
        if save_model_on_epoch:
            torch.save(
                model.state_dict(),
                os.path.join(output_dir, f"{output_prefix}-{epoch}.pt"),
            )
    return model

In [8]:
dataset = CSVMIMIC("<|note|>", truncate=True, gpt2_type="gpt2")

In [9]:
gpt2_type = "gpt2"

In [10]:
YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [11]:
#3e-9, 3e-8, 3e-6, 3e-5, 3e-4, 3e-2
learning_rates = [3e-5, 3e-4, 3e-2, 3e-1] # [3e-9,3e-8, 3e-6, 3e-5, 3e-4, 3e-2]
batch_sizes= [8,16,32, 46] #[8,16,32,64]
warmup_steps= [2500, 5000, 8000] # [1000,2500, 5000, 8000]
starting = True
for warmupIndex in range(len(warmup_steps)):
    for batchIndex in range(len(batch_sizes)):
        for lrIndex in range(len(learning_rates)):
            if starting:
                lrIndex= lrIndex+1
            starting = False
            model = train(
                dataset,
                GPT2LMHeadModel.from_pretrained(gpt2_type),
                GPT2Tokenizer.from_pretrained(gpt2_type),
                batch_size=batch_sizes[batchIndex],
                epochs=1,
                lr=learning_rates[lrIndex],
                max_seq_len=140,
                warmup_steps=warmup_steps[warmupIndex],
                gpt2_type=gpt2_type,
                device="cuda",
                output_dir="trained_models",
                output_prefix="note",
                save_model_on_epoch=False
            )
            modelDir = 'model'
            model.save_pretrained(modelDir)
            tokenizer=GPT2Tokenizer.from_pretrained(gpt2_type)
            tokenizer.save_pretrained(modelDir)
            result = !lm_eval \
            --model hf \
            --model_args pretrained=model \
            --output_path /Output \
            --include_path ./ \
            --tasks multimedqa \
            --device cpu \
            --trust_remote_code



            log_text = ' '.join(result)
            
            # Regular expression to match the accuracy value
            match = re.search(r'pubmedqa\s+\|[\s\S]+?\|([\d.]+)', log_text[250900:])

            if match:
                accuracy = float(match.group(1))
                print(f"Extracted Accuracy: {accuracy}")
            else:
                accuracy = print("Accuracy not found in log.")


            shutil.rmtree('model')
            with open("results", "a") as f:
                f.write(f'{{ "learningRate": {learning_rates[lrIndex]}, "batchSize": {batch_sizes[batchIndex]}, "warmupSteps": {warmup_steps[warmupIndex]}, "accuracy": {accuracy} }}\n')



Training epoch 0


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
20000it [12:06, 27.54it/s]


Accuracy not found in log.
Training epoch 0


2317it [01:24, 27.54it/s]


KeyboardInterrupt: 